# Experiment Orchestrator - Classifier Selection

Este notebook orquesta el experimento completo de selección de clasificador CNN.

**Backbones a evaluar:** ResNet-18, EfficientNet-B0, DenseNet-121

**Método:** Few-shot con support set de **glaucoma exclusivamente** — **5 iteraciones** con seeds compartidas

**Restricción del support set:**  
En REFUGE/train solo hay **40 imágenes de glaucoma**. El support set es exclusivamente de glaucoma  
(clase clínica de interés). Por eso N=50/100 son imposibles → se usan **N=25, 30, 35**.

| N | Glaucoma usados | % del disponible |
|---|----------------|------------------|
| 25 | 25/40 | 62.5% |
| 30 | 30/40 | 75.0% |
| 35 | 35/40 | 87.5% |

**Estrategia de seeds:**  
En la iteración `i`, TODOS los backbones usan `seeds[i]` → mismas imágenes, comparación justa.

| Iteración | Seed | ResNet-18 | EfficientNet-B0 | DenseNet-121 |
|-----------|------|-----------|-----------------|---------------|
| 0 | 42 | seed=42 | seed=42 | seed=42 |
| 1 | 123 | seed=123 | seed=123 | seed=123 |
| 2 | 456 | seed=456 | seed=456 | seed=456 |
| 3 | 789 | seed=789 | seed=789 | seed=789 |
| 4 | 1024 | seed=1024 | seed=1024 | seed=1024 |

**Dimensiones:** Few-shot (F1-macro en val completo), Grad-CAM IoU, Computacional

## Celda 1: Setup

Instala dependencias e importa módulos.

In [ ]:
# Instalar dependencias
# !pip install torch torchvision timm scikit-learn scipy pillow numpy pyyaml matplotlib seaborn

import sys
import json
import os
import numpy as np
sys.path.insert(0, '..')

from modules.data_module import DataModule
from modules.cnn_classifier import CNNClassifier
from scripts.few_shot import train_few_shot
from scripts.extract_gradcam import evaluate_gradcam_dataset
from scripts.benchmark_inference import run_benchmark

import yaml
import torch

# Cargar configuración
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

SEEDS     = config['few_shot']['seeds']     # [42, 123, 456, 789, 1024]
N_SAMPLES = config['few_shot']['n_samples'] # [25, 30, 35]
BACKBONES = config['backbones']             # ['resnet18', 'efficientnet_b0', 'densenet121']
N_KEYS    = [f'N{n}' for n in N_SAMPLES]   # ['N25', 'N30', 'N35']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Seeds: {SEEDS}')
print(f'N_samples (glaucoma only): {N_SAMPLES}')
print(f'Backbones: {BACKBONES}')

## Celda 2: Convert Data

Convierte el dataset REFUGE al formato del proyecto.

In [ ]:
# Convertir REFUGE → annotations.json + splits.json
import subprocess
subprocess.run(['python', '../scripts/convert_refuge_format.py'], check=True)

## Celda 3: Initialize DataModule

Carga los datos. El DataModule maneja la lógica de qué imágenes son glaucoma.

In [ ]:
# Inicializar DataModule (sin seed aquí — el seed va en few_shot por iteración)
data_module = DataModule(config['data'])
val_loader  = data_module.get_val_loader()

# Verificar que el split de train tiene exactamente 40 glaucoma
train_glaucoma = data_module.get_glaucoma_indices(split='train')
print(f'Val: {len(val_loader.dataset)} muestras (glaucoma + normal)')
print(f'Glaucoma disponibles en train: {len(train_glaucoma)} (máximo N factible = {len(train_glaucoma)-1})')
assert len(train_glaucoma) >= max(N_SAMPLES), \
    f'ERROR: Solo hay {len(train_glaucoma)} glaucoma pero se pide N={max(N_SAMPLES)}'

## Celda 4: Run Few-Shot Experiment — 5 Iteraciones

**Bucle externo:** 5 seeds.  
**Bucle interno:** 3 backbones, todos con **la misma seed** de esa iteración.

El support set es **exclusivamente glaucoma** (N=25, 30, 35 muestras de las 40 disponibles).  
La evaluación se hace en el **val set completo** (40 glaucoma + 360 normal).

**ADVERTENCIA:** Esta celda puede tomar varias horas (5 seeds × 3 backbones × 3 tamaños).

In [ ]:
# Estructura: results[backbone][seed] = {few_shot: {N25, N30, N35}, gradcam}
results = {backbone: {} for backbone in BACKBONES}

for seed in SEEDS:
    print(f'\n{"="*65}')
    print(f'ITERACIÓN seed={seed}  |  Todos los backbones usarán esta seed')
    print(f'{"="*65}')

    for backbone in BACKBONES:
        print(f'\n  ── Backbone: {backbone} (seed={seed})')

        # [1/2] Few-shot training: N=25, N=30, N=35 (glaucoma only)
        # train_few_shot muestrea glaucoma_ids internamente con la seed dada
        print(f'    [1/2] Few-shot training (N={N_SAMPLES}, glaucoma only)...')
        few_shot_result = train_few_shot(backbone, data_module, config, seed=seed)
        # few_shot_result = {"seed": seed, "backbone": backbone,
        #                    "N25": {f1_macro, accuracy, epochs_trained},
        #                    "N30": {...}, "N35": {...}}

        # [2/2] Grad-CAM con el modelo entrenado en N=35 (el mayor) de esta iteración
        # Se usa N=35 porque tiene más datos y por tanto mejor calidad de representación
        print('    [2/2] Evaluando Grad-CAM (modelo N35)...')
        model_path = f'../results/{backbone}/seed_{seed}/model_N35.pth'
        model = CNNClassifier({'backbone': backbone,
                               'num_classes': config['classifier']['num_classes'],
                               'pretrained': False,
                               'seed': seed})
        model.load(model_path)
        gradcam_metrics = evaluate_gradcam_dataset(model, val_loader, device)

        # Guardar resultados de esta iteración
        results[backbone][seed] = {
            'few_shot': few_shot_result,
            'gradcam':  gradcam_metrics,
        }
        print(f'    ✓ seed={seed} backbone={backbone} completado')

# Benchmark computacional (independiente de seed — mide hardware, no datos)
print(f'\n{"="*65}')
print('BENCHMARK COMPUTACIONAL (una sola vez por backbone)')
print(f'{"="*65}')
computational = {}
for backbone in BACKBONES:
    # Usar el modelo seed=42, N=35 para medir VRAM y tiempo
    model_path = f'../results/{backbone}/seed_42/model_N35.pth'
    model = CNNClassifier({'backbone': backbone,
                           'num_classes': config['classifier']['num_classes'],
                           'pretrained': False,
                           'seed': 42})
    model.load(model_path)
    computational[backbone] = run_benchmark(backbone, model, val_loader, device)
    print(f'  ✓ {backbone}: {computational[backbone]["total_parameters"]/1e6:.1f}M params, '
          f'{computational[backbone]["vram_batch_1_mb"]:.0f} MB VRAM')

## Celda 5: Agregar Resultados y Determinar Winner

Calcula `mean ± std` sobre las 5 seeds para cada backbone.  
El score de selección se calcula sobre las **medias**.

**Fórmula:**
```
Score = 0.40 × mean(F1@N25, F1@N30, F1@N35) + 0.40 × IoU_GradCAM + 0.20 × (1 - VRAM_norm)
```
Todas las métricas son la media sobre las 5 iteraciones.

In [ ]:
import numpy as np

summary = {}

for backbone in BACKBONES:
    seed_results = results[backbone]  # dict: {seed: {few_shot, gradcam}}

    # Recoger métricas por seed para cada N
    f1_by_n = {nk: [seed_results[s]['few_shot'][nk]['f1_macro'] for s in SEEDS]
               for nk in N_KEYS}
    iou_list = [seed_results[s]['gradcam']['mean_iou'] for s in SEEDS]

    # Construir per_seed
    per_seed = {}
    for s in SEEDS:
        per_seed[str(s)] = {
            'mean_iou_gradcam': seed_results[s]['gradcam']['mean_iou'],
        }
        for nk in N_KEYS:
            per_seed[str(s)][f'f1_{nk}'] = seed_results[s]['few_shot'][nk]['f1_macro']

    # Construir aggregated
    aggregated = {
        'mean_iou_mean': float(np.mean(iou_list)),
        'mean_iou_std':  float(np.std(iou_list)),
    }
    for nk in N_KEYS:
        aggregated[f'f1_{nk}_mean'] = float(np.mean(f1_by_n[nk]))
        aggregated[f'f1_{nk}_std']  = float(np.std(f1_by_n[nk]))

    summary[backbone] = {
        'per_seed':      per_seed,
        'aggregated':    aggregated,
        'computational': computational[backbone],
    }

# Calcular score sobre medias
all_vrams = [summary[b]['computational']['vram_batch_1_mb'] for b in BACKBONES]
scores = {}
for backbone in BACKBONES:
    agg  = summary[backbone]['aggregated']
    vram = summary[backbone]['computational']['vram_batch_1_mb']

    # Media de F1 sobre todos los N (N25, N30, N35)
    f1_mean  = float(np.mean([agg[f'f1_{nk}_mean'] for nk in N_KEYS]))
    iou_mean = agg['mean_iou_mean']
    vram_norm = (vram - min(all_vrams)) / (max(all_vrams) - min(all_vrams) + 1e-8)

    score = 0.40 * f1_mean + 0.40 * iou_mean + 0.20 * (1 - vram_norm)
    summary[backbone]['score'] = float(score)
    scores[backbone] = score

winner = max(scores, key=scores.get)

# Tabla de resultados
print(f'\nRESULTADOS FINALES (mean ± std sobre {len(SEEDS)} seeds):')
header = f'{"Backbone":<20}' + ''.join(f'  {nk:>12}' for nk in N_KEYS) + f'  {"IoU":>12}  {"Score":>8}'
print(header)
print('-' * len(header))
for b in BACKBONES:
    a = summary[b]['aggregated']
    row = f'{b:<20}'
    for nk in N_KEYS:
        row += f"  {a[f'f1_{nk}_mean']:.3f}±{a[f'f1_{nk}_std']:.3f}"
    row += f"  {a['mean_iou_mean']:.3f}±{a['mean_iou_std']:.3f}"
    row += f"  {summary[b]['score']:.4f}"
    print(row)
print(f'\n🏆 Winner: {winner} (score={scores[winner]:.4f})')

# Guardar selection_summary.json
import datetime
final = {
    'backbones':   summary,
    'winner':      winner,
    'seeds_used':  SEEDS,
    'n_samples':   N_SAMPLES,
    'config_used': config,
    'timestamp':   datetime.datetime.utcnow().isoformat() + 'Z',
}
os.makedirs('../results', exist_ok=True)
with open('../results/selection_summary.json', 'w') as f:
    json.dump(final, f, indent=2)
print('\n✓ Guardado: results/selection_summary.json')